In [1]:
import math
import csv
import numpy as np
import matplotlib.pyplot as plt
from math import e
import statistics as st
import cmath
import scipy
import scipy.stats as ss
import random
import seaborn as sns
import pandas as pd
import pyswarms
import emcee
import corner

In [8]:
data = map(lambda x: [ x[0], x[1], x[2] ],
        np.loadtxt("/Users/gopipatel/Documents/RRI_VSP/saras3_S11_200mm_above_water.s1p", skiprows=5))
d=list(data) #List of the format [Frequency, Magnitude, Phase (in degrees)]
v=[d[i][0] for i in range(len(d)-1)]
freq= [d[i][0] for i in range(len(d))]

df=pd.read_csv('/Users/gopipatel/Downloads/S11_for_freq.csv')
rlc = df[df.columns[1]].values.tolist()

In [9]:
def Compute_Tmeas_(PA, p0, p1, p2, p3, p4, p5, p6, PN, gamma_N, f, l): 
    A=0
    B=0
    C=0
    l=l
    itr=3
    Tmeas=[]
    phi_A=0
    phi_N=0
    phi_f=0
    P_ref=300
    c=3*1e8
    l=l
    N=7
    freq= [d[i][0] for i in range(len(d))]
    for i in range(len(v)):
        gamma_A=p0*1e-48* freq[i] ** (N - 1) + p1*1e-39* freq[i] ** (N - 2) + p2*1e-31* freq[i] ** (N - 3) + p3*e-23* freq[i] ** (N - 4) + p4*e-15* freq[i] ** (N - 5) + p5*1e-08* freq[i] ** (N - 6) + p6*1e-01
        
        phi= (4*math.pi*(freq[i])*l)/(0.7*c)
        A=sum((abs(gamma_A)**k)*(abs(gamma_N)**k)*sum(math.cos((2*l-k)*(phi_N+phi_A+phi))for l in range(k+1)) for k in range(itr))   
        
        B=sum(2*abs(f)*(abs(gamma_A)**(o+1))*(abs(gamma_N)**o)*math.cos(phi_f+(o+1)*(phi_A+phi)+o*phi_N) for o in range(itr))
    
        C=sum((abs(gamma_A)**b)*(abs(gamma_N)**b)*sum(math.cos((2*c-b)*(phi_N+phi_A+phi)) for c in range(b+1)) for b in range(itr))
        
        Tmeas.append((PA*A-P_ref)+PN*(B+(abs(f)**2)*(abs(gamma_A)**2)*C))
    return(Tmeas)

In [14]:
def chi_squared(params):
    T_model = Compute_Tmeas_(*params)+np.random.normal(0,0.001,70) #adding Gaussian Noise #Observed
    s=0.001                            #uncertainity
    chi2 = np.sum([((TA_exp[:] - T_model[:])/s)** 2 ])
    #chi2 = np.sum([((TA_exp[i] - T_model[i])/s)** 2  for i in range(len(freq))]) #least square fitting
    return chi2

In [15]:
#Basin Hoping

#args = (x1,y1,e1,sig_ss,pd['domain'], pd['additive'], pd['joint'], do_opt, met)
OPTIONS={'ftol':1e-6, 'xtol': 1e-6, 'maxiter':1e5, 'maxfev':1e5}

random.seed(16)
PA=random.randrange(0,500)
p0=random.uniform(-6,-5)
p1=random.uniform(2,3)
p2=random.uniform(-5,-4)
p3=random.uniform(4,5)
p4=random.uniform(-2.5,-2.0)
p5=random.uniform(5,6)
p6=random.uniform(4,5)
PN=random.randrange(50,150)
gamma_N=random.uniform(0,1)
f=random.uniform(0.05,0.5)
l=random.uniform(0.05,3)
#x0=[PA,gamma_A,PN,gamma_N,f,l] #initial value
x0=[300,-6.274954869166671, 2.716009717976181, -4.6891133131425535, 4.149936149195398, -2.270563185191849, 4.867143968582065, 4.753308867794712, 70.1, 0.29, 0.099, 1.9]
bounds=((0,500),(-6,-5),(2,3),(-5,-4),(4,5),(-2.5,-2.0),(5,6),(4,5),(50,150),(0,1),(0.05,0.5),(0.05,3)) #to return acceptable range for the params: PN, gamma_N, f, l
#minimizer_kwargs = {"method":"Nelder-Mead", 'options':OPTIONS, "bounds": bounds}

#result=scipy.optimize.basinhopping(chi_squared, x0, niter=200, T=1, stepsize=0.05, minimizer_kwargs=minimizer_kwargs)

p00 = x0

minimizer_kwargs={"method":"Nelder-Mead", 'options':OPTIONS, "bounds": bounds}

#for i in range(3):

   #print("\nPolynomial Order : {}".format(i+1))

for i in range (50):
    print(i+1)
    pout = scipy.optimize.basinhopping(chi_squared, p00, minimizer_kwargs=minimizer_kwargs,T=1e-5, stepsize=1e-5, niter=2, seed=15)
    p00 = pout.x
print(p00)

1


NameError: name 'TA_exp' is not defined